# 02 · Chunking
Turns `data/ingested/*.json` (from 01) into retrieval chunks in `data/chunks/chunks.jsonl`.

Rules:
- Token-based, about 300–400 tokens (approx tokens = chars / 4).
- Slides: one slide = one chunk. Consecutive slides with the same title are merged while the result stays within the limit. Only oversized slides are split.
- Non-slide sources (pasted notes): blocks are grouped under their headings and packed up to the limit.
- Code blocks and tables are never split. Only a code block or table that is bigger than a whole chunk is split, code by lines and tables by rows (repeating the header row).
- Every chunk starts with its slide or section title.

## 1 · Imports and settings

In [1]:
import difflib, hashlib, json, random, re, statistics
from collections import Counter, defaultdict
from pathlib import Path

DATA_DIR = Path("../data")
IN_DIR = DATA_DIR / "ingested"
OUT_DIR = DATA_DIR / "chunks"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKER_VERSION = "2"
MAX_TOKENS = 400            # upper bound for a chunk, title included
MIN_BODY_CHARS = 15         # slides with less body text than this (and no code) are dropped
TITLE_SIZE_RATIO = 1.2      # no heading block -> leading blocks this much larger than body font form the title
VISION_MAX_CHARS = 6000     # more vision-OCR text than this on one slide = runaway generation, quarantined
DUP_RATIO = 0.95            # blocks on the same page this similar are repeats from OCR; keep the first

SETTINGS = dict(version=CHUNKER_VERSION, max_tokens=MAX_TOKENS, min_body_chars=MIN_BODY_CHARS,
                title_size_ratio=TITLE_SIZE_RATIO, vision_max_chars=VISION_MAX_CHARS, dup_ratio=DUP_RATIO)
SETTINGS_HASH = hashlib.sha1(json.dumps(SETTINGS, sort_keys=True).encode()).hexdigest()[:12]
print("settings", SETTINGS, "hash", SETTINGS_HASH)

settings {'version': '2', 'max_tokens': 400, 'min_body_chars': 15, 'title_size_ratio': 1.2, 'vision_max_chars': 6000, 'dup_ratio': 0.95} hash eebcd81a8021


## 2 · Helpers: tokens, block rendering, titles

In [2]:
def n_tokens(text):
    return max(1, round(len(text) / 4))


def clean_title(text):
    return re.sub(r"\s+", " ", text).strip(" :-–•")


def title_key(title):
    """Titles that differ only by case or a 'contd' marker count as the same title."""
    t = title.lower()
    t = re.sub(r"\(?\s*(cont(inued|d)?\.?|\d+\s*/\s*\d+)\s*\)?$", "", t)
    return re.sub(r"[^a-z0-9]+", " ", t).strip()


BULLET_GLYPHS = re.compile(r"^[\x00\uf000-\uf8ff]\s*", flags=re.M)   # PDF symbol-font bullets at line start
JUNK_CHARS = re.compile(r"[\x00-\x08\x0b-\x1f\uf000-\uf8ff]")


def clean_text(text):
    return JUNK_CHARS.sub("", BULLET_GLYPHS.sub("\u2022 ", text))


def render_block(b):
    """Text of a block as it appears in a chunk. Code is fenced so the LLM sees it as code."""
    text = clean_text(b["text"]).strip("\n")
    if b.get("is_code"):
        return f"```\n{text}\n```"
    return text.strip()


def is_atomic(b):
    return bool(b.get("is_code") or b.get("is_table"))


def clean_blocks(page, log):
    """Drop OCR faults from 01: runaway vision output and repeated blocks. Records what was dropped in log."""
    blocks = [b for b in page["blocks"] if b["text"].strip()]
    vision = [b for b in blocks if b.get("ocr") == "vision" and not b.get("is_heading")]
    vision_chars = sum(len(b["text"]) for b in vision)
    if vision_chars > VISION_MAX_CHARS:
        log.append(("runaway vision", page["page_no"], len(vision), vision_chars))
        blocks = [b for b in blocks if b not in vision]
    kept = []
    for b in blocks:
        norm = re.sub(r"\s+", " ", b["text"]).strip()
        if any(difflib.SequenceMatcher(None, norm, k).ratio() >= DUP_RATIO
               for k in (re.sub(r"\s+", " ", x["text"]).strip() for x in kept) if abs(len(k) - len(norm)) < 0.1 * len(norm) + 5):
            log.append(("duplicate block", page["page_no"], 1, len(b["text"])))
            continue
        kept.append(b)
    return kept


def page_title(blocks, body_font):
    """Return (title, body_blocks). Heading blocks become the title; without them,
    leading blocks in a clearly larger font do (title slides)."""
    heads = [b for b in blocks if b.get("is_heading")]
    if heads:
        title = clean_title(" ".join(b["text"] for b in heads))
        return title, [b for b in blocks if not b.get("is_heading")]
    lead = []
    for b in blocks:
        if is_atomic(b) or not b["size"] or b["size"] < body_font * TITLE_SIZE_RATIO or len(b["text"]) > 120:
            break
        lead.append(b)
    if lead:
        return clean_title(" ".join(b["text"] for b in lead)), blocks[len(lead):]
    return "", blocks

## 3 · Splitting units that are too big
Only used when a single block is bigger than a whole chunk. Code splits at line boundaries, tables at row boundaries with the header row repeated, text at lines then sentences.

In [3]:
def split_lines(lines, budget, joiner="\n", header=None):
    pieces, cur = [], []
    head = [header] if header else []
    for line in lines:
        trial = joiner.join(head + cur + [line])
        if cur and n_tokens(trial) > budget:
            pieces.append(joiner.join(head + cur))
            cur = []
        cur.append(line)
    if cur:
        pieces.append(joiner.join(head + cur))
    return pieces


def split_block(b, budget):
    """Split one oversized block into rendered pieces that each fit the budget."""
    text = b["text"].strip("\n")
    if b.get("is_code"):
        inner = budget - 2                                   # room for the ``` fences
        return [f"```\n{p}\n```" for p in split_lines(text.split("\n"), inner)]
    if b.get("is_table"):
        rows = text.split("\n")
        return split_lines(rows[1:], budget, header=rows[0])
    lines = [l for l in text.split("\n") if l.strip()]
    if all(n_tokens(l) <= budget for l in lines):
        return split_lines(lines, budget)
    sentences = re.split(r"(?<=[.!?])\s+", " ".join(lines))
    out = []
    for s in sentences:                                       # last resort: hard cut very long sentences
        out += [s[i:i + budget * 4] for i in range(0, len(s), budget * 4)]
    return split_lines(out, budget, joiner=" ")

## 4 · Packing blocks into chunks
`pack` fills chunks with whole rendered blocks up to the budget. A code block or table is never cut to fill the remaining space; it starts a new chunk instead.

In [4]:
def pack(blocks, budget):
    """Return a list of (body_text, has_code, has_table)."""
    units = []
    for b in blocks:
        r = render_block(b)
        if n_tokens(r) <= budget:
            units.append((r, b.get("is_code", False), b.get("is_table", False)))
        else:
            units += [(p, b.get("is_code", False), b.get("is_table", False)) for p in split_block(b, budget)]
    chunks, cur = [], []
    for u in units:
        trial = "\n".join(x[0] for x in cur + [u])
        if cur and n_tokens(trial) > budget:
            chunks.append(cur)
            cur = []
        cur.append(u)
    if cur:
        chunks.append(cur)
    return [("\n".join(x[0] for x in c), any(x[1] for x in c), any(x[2] for x in c)) for c in chunks]


def budget_for(title):
    return MAX_TOKENS - n_tokens(title + "\n\n")

## 5 · Chunking a source
Slides are chunked one slide at a time, then same-title neighbours are merged. Non-slide pages are grouped into sections by their headings.

In [5]:
def slide_pieces(doc, log):
    """One or more raw pieces per slide: dicts with title, body, pages, flags."""
    pieces, dropped, prev_title = [], [], ""
    for page in doc["pages"]:
        title, body_blocks = page_title(clean_blocks(page, log), doc["body_font_size"])
        title = title or prev_title or Path(doc["name"]).stem
        prev_title = title
        body_chars = sum(len(b["text"].strip()) for b in body_blocks)
        body_key = title_key(" ".join(b["text"] for b in body_blocks))
        squash = lambda s: s.replace(" ", "")
        only_repeats_title = body_key and squash(body_key) in squash(title_key(doc["name"]) + title_key(title))
        if not any(is_atomic(b) for b in body_blocks) and (body_chars < MIN_BODY_CHARS or only_repeats_title):
            dropped.append((page["page_no"], title))
            continue
        parts = pack(body_blocks, budget_for(title))
        for body, has_code, has_table in parts:
            pieces.append(dict(title=title, body=body, page_start=page["page_no"], page_end=page["page_no"],
                               has_code=has_code, has_table=has_table, whole_slide=len(parts) == 1))
    return pieces, dropped


def merge_same_title(pieces):
    """Merge consecutive whole slides that share a title while they fit in one chunk."""
    out = []
    for p in pieces:
        last = out[-1] if out else None
        if (last and last["whole_slide"] and p["whole_slide"]
                and title_key(last["title"]) == title_key(p["title"])
                and n_tokens(last["body"] + "\n\n" + p["body"]) <= budget_for(last["title"])):
            last["body"] += "\n\n" + p["body"]
            last["page_end"] = p["page_end"]
            last["has_code"] |= p["has_code"]
            last["has_table"] |= p["has_table"]
        else:
            out.append(dict(p))
    return out


def section_pieces(doc, log):
    """Non-slide sources: a heading starts a new section; sections are packed to the budget."""
    sections, cur = [], None
    for page in doc["pages"]:
        for b in clean_blocks(page, log):
            if b.get("is_heading") or cur is None:
                title = clean_title(b["text"]) if b.get("is_heading") else Path(doc["name"]).stem
                cur = dict(title=title, blocks=[], page_start=page["page_no"], page_end=page["page_no"])
                sections.append(cur)
                if b.get("is_heading"):
                    continue
            cur["blocks"].append(b)
            cur["page_end"] = page["page_no"]
    pieces = []
    for s in sections:
        for body, has_code, has_table in pack(s["blocks"], budget_for(s["title"])):
            pieces.append(dict(title=s["title"], body=body, page_start=s["page_start"], page_end=s["page_end"],
                               has_code=has_code, has_table=has_table, whole_slide=False))
    return pieces, []


def chunk_doc(doc, log):
    is_slides = any(p["is_slide"] for p in doc["pages"])
    pieces, dropped = slide_pieces(doc, log) if is_slides else section_pieces(doc, log)
    if is_slides:
        pieces = merge_same_title(pieces)
    chunks = []
    for i, p in enumerate(pieces):
        text = f"{p['title']}\n\n{p['body']}"
        chunks.append(dict(
            chunk_id=f"{doc['source_id']}-{i:04d}", source_id=doc["source_id"], source_name=doc["name"],
            role=doc["role"], page_start=p["page_start"], page_end=p["page_end"], title=p["title"],
            has_code=p["has_code"], has_table=p["has_table"], n_tokens=n_tokens(text), text=text,
            doc_hash=doc["hash"], chunk_settings=SETTINGS_HASH))
    return chunks, dropped

## 6 · Run on all ingested sources and save

In [6]:
def load_docs():
    return [json.loads(f.read_text(encoding="utf-8")) for f in sorted(IN_DIR.glob("*.json"))]


def chunk_all(docs):
    chunks, dropped, cleaned = [], {}, {}
    for d in docs:
        log = []
        c, dr = chunk_doc(d, log)
        chunks += c
        dropped[d["name"]], cleaned[d["name"]] = dr, log
    return chunks, dropped, cleaned


def save_chunks(chunks):
    with open(OUT_DIR / "chunks.jsonl", "w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")
    (OUT_DIR / "settings.json").write_text(json.dumps(dict(SETTINGS, hash=SETTINGS_HASH), indent=2))


docs = load_docs()
chunks, dropped, cleaned = chunk_all(docs)
save_chunks(chunks)
print(f"{len(docs)} sources -> {len(chunks)} chunks -> {OUT_DIR / 'chunks.jsonl'}")

11 sources -> 476 chunks -> ..\data\chunks\chunks.jsonl


---
# Checks

## 7 · Chunk count and token sizes per source

In [7]:
def size_table(chunks):
    by_src = defaultdict(list)
    for c in chunks:
        by_src[c["source_name"]].append(c["n_tokens"])
    bins = [(0, 50), (50, 100), (100, 200), (200, 300), (300, 400), (400, 10**9)]
    head = f"{'source':48} {'role':8} {'chunks':>6} {'min':>4} {'med':>4} {'mean':>5} {'max':>4}  " + " ".join(f"{a}-{b if b < 10**9 else '':<3}" for a, b in bins)
    print(head); print("-" * len(head))
    for name, toks in sorted(by_src.items()):
        role = next(c["role"] for c in chunks if c["source_name"] == name)
        hist = " ".join(f"{sum(a <= t < b for t in toks):>7}" for a, b in bins)
        print(f"{name[:48]:48} {role:8} {len(toks):>6} {min(toks):>4} {statistics.median(toks):>4.0f} {statistics.mean(toks):>5.0f} {max(toks):>4}  {hist}")
    allt = [c["n_tokens"] for c in chunks]
    print(f"\nALL: {len(allt)} chunks, median {statistics.median(allt):.0f}, mean {statistics.mean(allt):.0f}, max {max(allt)}; "
          f"over {MAX_TOKENS}: {sum(t > MAX_TOKENS for t in allt)}")
    print(f"multi-page chunks (merged slides): {sum(c['page_end'] > c['page_start'] for c in chunks)}, "
          f"with code: {sum(c['has_code'] for c in chunks)}, with table: {sum(c['has_table'] for c in chunks)}")


size_table(chunks)

source                                           role     chunks  min  med  mean  max  0-50  50-100 100-200 200-300 300-400 400-   
-----------------------------------------------------------------------------------------------------------------------------------
Ch 1 Data Mining Basics.pdf                      textbook     46   22  127   154  374        4      10      21       5       6       0
Ch 1 Machine Learning.pdf                        textbook     16   63  155   206  399        0       3       7       1       5       0
Ch 3. Classification.pdf                         textbook    129   10  110   145  378       17      38      39      21      14       0
Ch 4 Unsupervised Learning.pdf                   textbook     67   17  150   167  376        8      15      20      11      13       0
Ch 5 File Handling.pdf                           textbook     34   12   91   117  292        5      14      10       5       0       0
Ch 7. Regular Expressions.pdf                    textbook    

## 8 · What was removed
OCR faults from 01 (runaway vision output, repeated blocks), and slides with no real body text (title slides, "Demo" placeholders, slides whose only text repeats the title).

In [8]:
for name, log in cleaned.items():
    runaway = [x for x in log if x[0] == "runaway vision"]
    dups = [x for x in log if x[0] == "duplicate block"]
    if log:
        print(f"{name}: duplicate blocks removed {len(dups)} (pages {sorted({x[1] for x in dups})})"
              + "".join(f"; RUNAWAY VISION p{p}: {n} blocks, {ch} chars quarantined" for _, p, n, ch in runaway))
print()
for name, dr in dropped.items():
    if dr:
        print(f"{name}: {len(dr)} dropped ->", ", ".join(f"p{p} {t[:30]!r}" for p, t in dr[:8]), "..." if len(dr) > 8 else "")

Ch 3. Classification.pdf: duplicate blocks removed 6 (pages [93, 157])
Ch 5 File Handling.pdf: duplicate blocks removed 1 (pages [18])
Perceptron and LMS.pdf: duplicate blocks removed 1 (pages [30])
Ch1 and 2 Introduction to Python and Python Data Structure.pdf: duplicate blocks removed 2 (pages [93, 95])
Ch3. Control Flow_part1.pdf: duplicate blocks removed 1 (pages [8])
Ch3 Functions Modules and Packages_part2.pdf: duplicate blocks removed 5 (pages [17, 33, 39, 41])
Ch 1 Data Mining Basics.pdf: duplicate blocks removed 6 (pages [58, 59])
Ch 4 Unsupervised Learning.pdf: duplicate blocks removed 8 (pages [28, 29, 30, 60]); RUNAWAY VISION p65: 75 blocks, 12319 chars quarantined

Ch 3. Classification.pdf: 3 dropped -> p102 'T-test Python', p166 'Techniques to avoid the Overfi', p172 'Bayes Classification Methods' 
Ch 5 File Handling.pdf: 1 dropped -> p1 'Ch5. File Handling' 
Ch 1 Machine Learning.pdf: 1 dropped -> p1 'Ch 1 Machine Learning' 
Ch1 and 2 Introduction to Python and Python Da

## 9 · Five random chunks, printed in full

In [9]:
def show_chunk(c):
    pages = f"p{c['page_start']}" + (f"-{c['page_end']}" if c["page_end"] != c["page_start"] else "")
    print(f"=== {c['chunk_id']} | {c['source_name']} {pages} | {c['n_tokens']} tok | code={c['has_code']} table={c['has_table']}")
    print(c["text"]); print()


random.seed(7)
for c in random.sample(chunks, 5):
    show_chunk(c)

=== 4d4850eb7190-0001 | Perceptron and LMS.pdf p2 | 109 tok | code=False table=False
Principal of minimal disturbance

• Most of the supervised learning algorithms are based on this principal which
states:
• Adopt to reduce the output error for the current training pattern, with minimal
disturbance to responses already learned.
• If a learning procedure ignores this principle, the encoding of new patterns
might perturb the system into new weight states that cause information concerning older pattern to be forgotten.

=== 0e0fda908c31-0077 | Ch 3. Classification.pdf p120 | 31 tok | code=False table=False
Interpreting Regression Results

• Coefficients and their significance
• R-squared and adjusted R-squared
• Residual analysis

=== 757f941ce10b-0002 | Ch1 and 2 Introduction to Python and Python Data Structure.pdf p5 | 50 tok | code=False table=False
Downside

• Execution may not be as fast as C and C++
– Due to byte code interpretation
• But Speed-of-development gain is far more import

## 10 · Proof that no code block or table was split
For every code or table block in the ingested data:
- if it fits in one chunk, its exact text must appear **whole in exactly one chunk** covering its page;
- if it is bigger than a chunk (allowed to split), the pieces must lie in consecutive chunks of the same source and join back, line for line, into the original.

Only chunks covering the block's own page count, because some decks repeat the same table on several slides. Also checks that every other body block's text made it into some chunk.

In [10]:
def fenced_segments(text):
    return re.findall(r"```\n(.*?)\n```", text, flags=re.S)


def verify_atomic(docs, chunks):
    by_src = defaultdict(list)
    for c in chunks:
        by_src[c["source_id"]].append(c)
    stats = Counter()
    problems = []
    for d in docs:
        dropped_pages = {p for p, _ in dropped.get(d["name"], [])}
        for page in d["pages"]:
            if page["page_no"] in dropped_pages:
                continue
            cs = [c for c in by_src[d["source_id"]] if c["page_start"] <= page["page_no"] <= c["page_end"]]
            for b in clean_blocks(page, []):
                if b.get("is_heading"):
                    continue
                kind = "code" if b.get("is_code") else "table" if b.get("is_table") else "text"
                r = render_block(b)
                hits = [c for c in cs if r in c["text"]]
                if kind == "text":
                    ok = bool(hits) or all(any(l.strip() in c["text"] for c in cs) for l in r.split("\n") if l.strip())
                    stats["text_found" if ok else "text_missing"] += 1
                    if not ok:
                        problems.append(("text missing", d["name"], page["page_no"], b["text"][:60]))
                    continue
                if len(hits) == 1:
                    stats[f"{kind}_whole"] += 1
                elif len(hits) > 1:
                    stats[f"{kind}_duplicated"] += 1
                    problems.append((f"{kind} in {len(hits)} chunks", d["name"], page["page_no"], b["text"][:60]))
                else:
                    original = clean_text(b["text"]).strip("\n")
                    if kind == "code":
                        idx = [i for i, c in enumerate(cs) if any(seg and seg in original for seg in fenced_segments(c["text"]))]
                        joined = "\n".join(seg for i in idx for seg in fenced_segments(cs[i]["text"]) if seg in original)
                    else:
                        rows = original.split("\n")
                        idx = [i for i, c in enumerate(cs) if rows[0] in c["text"] and any(r_ in c["text"] for r_ in rows[1:])]
                        joined = "\n".join([rows[0]] + [r_ for i in idx for r_ in cs[i]["text"].split("\n") if r_ in rows[1:]])
                    consecutive = idx == list(range(idx[0], idx[0] + len(idx))) if idx else False
                    oversized = n_tokens(r) > budget_for("")
                    if joined == original and consecutive and oversized:
                        stats[f"{kind}_split_by_lines_ok"] += 1
                    else:
                        stats[f"{kind}_BROKEN"] += 1
                        problems.append((f"{kind} broken", d["name"], page["page_no"], b["text"][:60]))
    return stats, problems


stats, problems = verify_atomic(docs, chunks)
for k, v in sorted(stats.items()):
    print(f"{k:28} {v}")
print("\nproblems:", len(problems))
for p in problems[:15]:
    print("  ", p)

code_whole                   253
table_whole                  38
text_found                   2194

problems: 0


## 11 · The 10 largest and 10 smallest chunks

In [11]:
def brief(c):
    pages = f"p{c['page_start']}" + (f"-{c['page_end']}" if c["page_end"] != c["page_start"] else "")
    body = c["text"].split("\n", 2)[-1].replace("\n", " / ")
    print(f"{c['n_tokens']:>4} tok  {c['source_name'][:30]:30} {pages:8} {c['title'][:35]!r:38} {body[:90]!r}")


ranked = sorted(chunks, key=lambda c: c["n_tokens"])
print("LARGEST")
for c in ranked[::-1][:10]:
    brief(c)
print("\nSMALLEST")
for c in ranked[:10]:
    brief(c)

LARGEST
 399 tok  Ch 1 Machine Learning.pdf      p17-20   'Types of Learning'                    '• Supervised machine learning / – It is defined by its use of labeled datasets to train / '
 396 tok  Perceptron and LMS.pdf         p8-12    'Perceptron Learning'                  '• Being a supervised learning, the weight updation (training) is done on the / basis of co'
 383 tok  Perceptron and LMS.pdf         p28-30   'Problem 3'                            'Input Weight changes weights x1 x2 x3 x4 b Yin Y t Δw1 Δw2 Δw3 Δw4 Δb w1 w2 w3 w4 b 0 0 0 '
 381 tok  Perceptron and LMS.pdf         p35-37   'Widrow- Hoff Rule'                    '• Widrow-Hoﬀ learning rule was proposed in 1960. / • It is a direct application of least-s'
 378 tok  Ch 3. Classification.pdf       p24-26   'Information Gain'                     'Class-Labeled Training Tuples from the AllElectronics Customer Database / RID | age | inco'
 376 tok  Ch 4 Unsupervised Learning.pdf p48-50   'Density-based methods'         

## 12 · Saved file

In [12]:
lines = (OUT_DIR / "chunks.jsonl").read_text(encoding="utf-8").splitlines()
first = json.loads(lines[0])
print(len(lines), "lines; keys:", list(first))
missing = [k for k in ["chunk_id", "source_id", "source_name", "role", "page_start", "page_end", "title", "has_code", "has_table"]
           if any(k not in json.loads(l) for l in lines)]
print("required metadata missing:", missing or "none")
print("duplicate chunk_ids:", len(lines) - len({json.loads(l)["chunk_id"] for l in lines}))

476 lines; keys: ['chunk_id', 'source_id', 'source_name', 'role', 'page_start', 'page_end', 'title', 'has_code', 'has_table', 'n_tokens', 'text', 'doc_hash', 'chunk_settings']
required metadata missing: none
duplicate chunk_ids: 0
